In [1]:
import pandas as pd
import numpy as np
import pickle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [2]:
d = pd.read_csv("final_file.csv")

In [3]:
# ============================================================
# NETTOYAGE DE BASE
# ============================================================
d['Prix'] = d['Prix'].str.replace(r'[^\d.]', '', regex=True).astype(float)
d['Kilométrage'] = d['Kilométrage'].str.replace(r'[^\d.]', '', regex=True).astype(float)
d['Puissance fiscale'] = d['Puissance fiscale'].str.extract(r'(\d+)').astype(float)

d = d.dropna(subset=['Prix', 'Année-Modèle'])

for col in ['Boite de vitesses', 'Type de carburant', 'Marque', 'Modèle', 'État']:
    d[col] = d[col].fillna(d[col].mode()[0])

d['Kilométrage'] = d['Kilométrage'].fillna(d['Kilométrage'].median())
d['Puissance fiscale'] = d['Puissance fiscale'].fillna(d['Puissance fiscale'].median())
d['Nombre de portes'] = d['Nombre de portes'].fillna(d['Nombre de portes'].median())
d['Origine'] = d['Origine'].fillna('Non spécifiée')

d = d.drop(columns=['Première main'], errors='ignore')
d = d.drop_duplicates()

In [4]:
# ============================================================
# ENCODAGE
# ============================================================
df = d.copy()

# État ordinal (logique : Pour Pièces < Endommagé < ... < Neuf)
df['État'] = df['État'].replace({
    'Pour Pièces': 0, 'Endommagé': 1, 'Correct': 2,
    'Bon': 3, 'Très bon': 4, 'Excellent': 5, 'Neuf': 6
}).astype(int)

In [5]:
# Boite de vitesses binaire
df['Boite de vitesses'] = df['Boite de vitesses'].map({'Manuelle': 0, 'Automatique': 1})

In [6]:
# One-hot encoding (Type carburant, Origine, Marque, Modèle)
df = pd.get_dummies(df, columns=['Type de carburant', 'Origine', 'Marque', 'Modèle'])

In [7]:
# Équipements
equipements = df['Équipements'].apply(eval)
for carac in equipements.explode().dropna().unique():
    df[carac] = equipements.apply(lambda x: carac in x)
df = df.drop(['Équipements'], axis=1)

df['Année-Modèle'] = pd.to_numeric(df['Année-Modèle'], errors='coerce')
df['Nombre de portes'] = df['Nombre de portes'].astype('int64')
df = df.dropna()

In [8]:
# ============================================================
# SUPPRESSION DES OUTLIERS (IQR)
# ============================================================
for col in ['Prix', 'Année-Modèle', 'Kilométrage', 'Puissance fiscale']:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    df = df[(df[col] >= Q1 - 1.5 * IQR) & (df[col] <= Q3 + 1.5 * IQR)]

print(f"Shape après nettoyage : {df.shape}")

Shape après nettoyage : (10479, 661)


In [9]:
# ============================================================
# FEATURE ENGINEERING (NOUVELLES COLONNES)
# ============================================================
df['Age'] = 2026 - df['Année-Modèle']
df['Km_par_an'] = df['Kilométrage'] / (df['Age'] + 1)

In [10]:
# Score équipements (somme des colonnes booléennes d'équipements)
equipements_cols = [col for col in df.columns if col in [
    'Climatisation', 'GPS', 'Caméra de recul', 'Jantes en alliage',
    'Toit ouvrant', 'Sièges cuir', 'Radar de recul',
    'Vitres électriques', 'Verrouillage centralisé',
    'Régulateur de vitesse', 'ABS', 'Airbags',
    'Direction assistée', 'CD / MP3 / Bluetooth'
]]
df['Score_equipements'] = df[equipements_cols].sum(axis=1)

print(f"Nouvelles features ajoutées : Age, Km_par_an, Score_equipements")

Nouvelles features ajoutées : Age, Km_par_an, Score_equipements


In [11]:

# ============================================================
# LOG-TRANSFORMATION DU PRIX (cible)
# ============================================================
df['Prix_log'] = np.log1p(df['Prix'])
print(f"Skewness Prix brut : {df['Prix'].skew():.3f}")
print(f"Skewness Prix log  : {df['Prix_log'].skew():.3f}")

Skewness Prix brut : 0.825
Skewness Prix log  : -0.921


In [12]:
# ============================================================
# SÉPARATION X / y — AVEC et SANS log pour comparer
# ============================================================
X = df.drop(columns=['Prix', 'Prix_log'])
y_brut = df['Prix']
y_log  = df['Prix_log']

In [13]:
# ============================================================
# TRAIN / TEST SPLIT — FAIT EN PREMIER
# ============================================================
X_train, X_test, y_train_brut, y_test_brut = train_test_split(
    X, y_brut, test_size=0.2, random_state=42
)
_, _, y_train_log, y_test_log = train_test_split(
    X, y_log, test_size=0.2, random_state=42
)


In [14]:
# ============================================================
# STANDARD SCALER — APRÈS LE SPLIT (✅ pas de data leakage)
# ============================================================
numerical_features = ['Année-Modèle', 'Kilométrage', 'Puissance fiscale', 'Age', 'Km_par_an']

scaler = StandardScaler()
X_train[numerical_features] = scaler.fit_transform(X_train[numerical_features])   # fit sur train SEULEMENT
X_test[numerical_features]  = scaler.transform(X_test[numerical_features])         # transform sur test


In [15]:
# Sauvegarde
pickle.dump(scaler, open('scaler_v2.pkl', 'wb'))
pickle.dump(X.columns.tolist(), open('features_v2.pkl', 'wb'))

print("\n✅ Étape 1 terminée :")
print(f"   X_train : {X_train.shape}")
print(f"   X_test  : {X_test.shape}")
print(f"   Features sauvegardées dans features_v2.pkl")
print(f"   Scaler sauvegardé dans scaler_v2.pkl")


✅ Étape 1 terminée :
   X_train : (8383, 663)
   X_test  : (2096, 663)
   Features sauvegardées dans features_v2.pkl
   Scaler sauvegardé dans scaler_v2.pkl


In [16]:
# Sauvegarde des données préparées pour l'étape 2
import pickle
with open('data_etape1.pkl', 'wb') as f:
    pickle.dump({
        'X_train': X_train, 'X_test': X_test,
        'y_train_brut': y_train_brut, 'y_test_brut': y_test_brut,
        'y_train_log': y_train_log, 'y_test_log': y_test_log
    }, f)

print("   Données sauvegardées dans data_etape1.pkl")


   Données sauvegardées dans data_etape1.pkl


In [17]:
import pandas as pd
import numpy as np
import pickle
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')


In [18]:
# ============================================================
# CHARGEMENT DES DONNÉES DE L'ÉTAPE 1
# ============================================================
with open('data_etape1.pkl', 'rb') as f:
    data = pickle.load(f)

X_train      = data['X_train']
X_test       = data['X_test']
y_train_brut = data['y_train_brut']
y_test_brut  = data['y_test_brut']
y_train_log  = data['y_train_log']
y_test_log   = data['y_test_log']

print(f"X_train : {X_train.shape} | X_test : {X_test.shape}")
print("="*60)


X_train : (8383, 663) | X_test : (2096, 663)


In [19]:
# ============================================================
# FONCTION D'ÉVALUATION UNIVERSELLE
# ============================================================
def evaluer(nom, y_test, y_pred, log_mode=False):
    """Calcule MAE, RMSE, R² — retransformation si log_mode=True"""
    if log_mode:
        y_pred = np.expm1(y_pred)
        y_test = np.expm1(y_test)
    mae  = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2   = r2_score(y_test, y_pred)
    print(f"\n{'='*60}")
    print(f"  {nom}")
    print(f"{'='*60}")
    print(f"  MAE  : {mae:>12,.0f} MAD")
    print(f"  RMSE : {rmse:>12,.0f} MAD")
    print(f"  R²   : {r2:>12.4f}")
    return {'nom': nom, 'mae': mae, 'rmse': rmse, 'r2': r2}

resultats = []

In [20]:
# ============================================================
# MODÈLE 1 — RandomForest de BASE (référence, ton modèle actuel)
# ============================================================
print("\n⏳ Modèle 1 : RandomForest de base (référence)...")
rf_base = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_base.fit(X_train, y_train_brut)
y_pred = rf_base.predict(X_test)
resultats.append(evaluer("RF Base (100 arbres, sans log)", y_test_brut, y_pred))


⏳ Modèle 1 : RandomForest de base (référence)...

  RF Base (100 arbres, sans log)
  MAE  :       16,402 MAD
  RMSE :       27,509 MAD
  R²   :       0.8889


In [21]:
# ============================================================
# MODÈLE 2 — RandomForest AMÉLIORÉ (300 arbres + nouvelles features)
# ============================================================
print("\n⏳ Modèle 2 : RandomForest amélioré (300 arbres)...")
rf_v2 = RandomForestRegressor(
    n_estimators=300,
    max_depth=None,
    max_features='sqrt',
    min_samples_leaf=1,
    min_samples_split=2,
    random_state=42,
    n_jobs=-1
)
rf_v2.fit(X_train, y_train_brut)
y_pred = rf_v2.predict(X_test)
resultats.append(evaluer("RF Amélioré (300 arbres, nouvelles features)", y_test_brut, y_pred))


⏳ Modèle 2 : RandomForest amélioré (300 arbres)...

  RF Amélioré (300 arbres, nouvelles features)
  MAE  :       19,775 MAD
  RMSE :       30,853 MAD
  R²   :       0.8603


In [22]:
# ============================================================
# MODÈLE 3 — RandomForest + LOG du prix
# ============================================================
print("\n⏳ Modèle 3 : RandomForest + log(Prix)...")
rf_log = RandomForestRegressor(
    n_estimators=300,
    max_features='sqrt',
    random_state=42,
    n_jobs=-1
)
rf_log.fit(X_train, y_train_log)
y_pred_log = rf_log.predict(X_test)
resultats.append(evaluer("RF + log(Prix)", y_test_log, y_pred_log, log_mode=True))


⏳ Modèle 3 : RandomForest + log(Prix)...

  RF + log(Prix)
  MAE  :       20,451 MAD
  RMSE :       33,014 MAD
  R²   :       0.8400


In [23]:
# ============================================================
# MODÈLE 4 — XGBoost BIEN CONFIGURÉ
# ============================================================
print("\n⏳ Modèle 4 : XGBoost bien configuré...")
xgb_model = xgb.XGBRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=30,
    eval_metric='mae'
)
xgb_model.fit(
    X_train, y_train_brut,
    eval_set=[(X_test, y_test_brut)],
    verbose=False
)
print(f"  Meilleur n_estimators : {xgb_model.best_iteration}")
y_pred = xgb_model.predict(X_test)
resultats.append(evaluer("XGBoost bien configuré", y_test_brut, y_pred))


⏳ Modèle 4 : XGBoost bien configuré...
  Meilleur n_estimators : 824

  XGBoost bien configuré
  MAE  :       15,466 MAD
  RMSE :       25,630 MAD
  R²   :       0.9036


In [24]:
# ============================================================
# MODÈLE 5 — XGBoost + LOG du prix
# ============================================================
print("\n⏳ Modèle 5 : XGBoost + log(Prix)...")
xgb_log = xgb.XGBRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=30,
    eval_metric='mae'
)
xgb_log.fit(
    X_train, y_train_log,
    eval_set=[(X_test, y_test_log)],
    verbose=False
)
y_pred_log = xgb_log.predict(X_test)
resultats.append(evaluer("XGBoost + log(Prix)", y_test_log, y_pred_log, log_mode=True))


⏳ Modèle 5 : XGBoost + log(Prix)...

  XGBoost + log(Prix)
  MAE  :       16,338 MAD
  RMSE :       27,714 MAD
  R²   :       0.8873


In [25]:
# ============================================================
# TABLEAU COMPARATIF FINAL
# ============================================================
print("\n" + "="*60)
print("  TABLEAU COMPARATIF FINAL")
print("="*60)
print(f"  {'Modèle':<45} {'MAE':>10} {'RMSE':>12} {'R²':>8}")
print(f"  {'-'*45} {'-'*10} {'-'*12} {'-'*8}")
for r in sorted(resultats, key=lambda x: x['r2'], reverse=True):
    print(f"  {r['nom']:<45} {r['mae']:>10,.0f} {r['rmse']:>12,.0f} {r['r2']:>8.4f}")


  TABLEAU COMPARATIF FINAL
  Modèle                                               MAE         RMSE       R²
  --------------------------------------------- ---------- ------------ --------
  XGBoost bien configuré                            15,466       25,630   0.9036
  RF Base (100 arbres, sans log)                    16,402       27,509   0.8889
  XGBoost + log(Prix)                               16,338       27,714   0.8873
  RF Amélioré (300 arbres, nouvelles features)      19,775       30,853   0.8603
  RF + log(Prix)                                    20,451       33,014   0.8400


In [26]:
# ============================================================
# SAUVEGARDE DU MEILLEUR MODÈLE
# ============================================================
meilleur = max(resultats, key=lambda x: x['r2'])
print(f"\n🏆 Meilleur modèle : {meilleur['nom']}")
print(f"   R² = {meilleur['r2']:.4f} | MAE = {meilleur['mae']:,.0f} MAD")


🏆 Meilleur modèle : XGBoost bien configuré
   R² = 0.9036 | MAE = 15,466 MAD


In [27]:
# Sauvegarde de tous les modèles pour l'étape 3
with open('modeles_etape2.pkl', 'wb') as f:
    pickle.dump({
        'rf_base': rf_base,
        'rf_v2': rf_v2,
        'rf_log': rf_log,
        'xgb_model': xgb_model,
        'xgb_log': xgb_log,
        'resultats': resultats
    }, f)

print("\n✅ Étape 2 terminée — modèles sauvegardés dans modeles_etape2.pkl")
print("   → Lance étape3_tuning.py pour l'optimisation fine")


✅ Étape 2 terminée — modèles sauvegardés dans modeles_etape2.pkl
   → Lance étape3_tuning.py pour l'optimisation fine


In [28]:
import pandas as pd
import numpy as np
import pickle
from sklearn.model_selection import RandomizedSearchCV, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import StackingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

In [29]:
# ============================================================
# CHARGEMENT
# ============================================================
with open('data_etape1.pkl', 'rb') as f:
    data = pickle.load(f)

X_train      = data['X_train']
X_test       = data['X_test']
y_train      = data['y_train_brut']
y_test       = data['y_test_brut']

print(f"X_train : {X_train.shape} | X_test : {X_test.shape}")

def evaluer(nom, y_test, y_pred):
    mae  = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2   = r2_score(y_test, y_pred)
    print(f"\n{'='*60}")
    print(f"  {nom}")
    print(f"{'='*60}")
    print(f"  MAE  : {mae:>12,.0f} MAD")
    print(f"  RMSE : {rmse:>12,.0f} MAD")
    print(f"  R²   : {r2:>12.4f}")
    return {'nom': nom, 'mae': mae, 'rmse': rmse, 'r2': r2}

resultats = []

X_train : (8383, 663) | X_test : (2096, 663)


In [30]:
# ============================================================
# RÉFÉRENCE — XGBoost de l'étape 2
# ============================================================
print("\n⏳ Référence : XGBoost étape 2 (sans tuning)...")
xgb_ref = xgb.XGBRegressor(
    n_estimators=1000, learning_rate=0.05, max_depth=6,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=1.0,
    random_state=42, n_jobs=-1,
    early_stopping_rounds=30, eval_metric='mae'
)
xgb_ref.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
y_pred = xgb_ref.predict(X_test)
resultats.append(evaluer("XGBoost référence (étape 2)", y_test, y_pred))


⏳ Référence : XGBoost étape 2 (sans tuning)...

  XGBoost référence (étape 2)
  MAE  :       15,466 MAD
  RMSE :       25,630 MAD
  R²   :       0.9036


In [31]:
# ============================================================
# RANDOMIZED SEARCH sur XGBoost
# ============================================================
print("\n⏳ RandomizedSearchCV sur XGBoost (patience ~10-20 min)...")

param_dist = {
    'n_estimators':      [500, 700, 1000, 1200],
    'learning_rate':     [0.01, 0.03, 0.05, 0.07],
    'max_depth':         [4, 5, 6, 7, 8],
    'subsample':         [0.7, 0.8, 0.9],
    'colsample_bytree':  [0.6, 0.7, 0.8, 0.9],
    'colsample_bylevel': [0.6, 0.8, 1.0],
    'reg_alpha':         [0, 0.01, 0.1, 0.5, 1.0],
    'reg_lambda':        [0.5, 1.0, 2.0, 5.0],
    'min_child_weight':  [1, 3, 5, 7],
    'gamma':             [0, 0.1, 0.2, 0.5]
}

xgb_search = xgb.XGBRegressor(
    random_state=42, n_jobs=-1,
    eval_metric='mae'
)

cv = KFold(n_splits=5, shuffle=True, random_state=42)

search = RandomizedSearchCV(
    xgb_search,
    param_distributions=param_dist,
    n_iter=40,               # 40 combinaisons aléatoires
    cv=cv,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    random_state=42,
    verbose=1
)

search.fit(X_train, y_train)

print(f"\n  Meilleurs paramètres trouvés :")
for k, v in search.best_params_.items():
    print(f"    {k} = {v}")

best_xgb = search.best_estimator_
y_pred = best_xgb.predict(X_test)
resultats.append(evaluer("XGBoost tuné (RandomizedSearch)", y_test, y_pred))



⏳ RandomizedSearchCV sur XGBoost (patience ~10-20 min)...
Fitting 5 folds for each of 40 candidates, totalling 200 fits

  Meilleurs paramètres trouvés :
    subsample = 0.8
    reg_lambda = 2.0
    reg_alpha = 1.0
    n_estimators = 700
    min_child_weight = 1
    max_depth = 8
    learning_rate = 0.07
    gamma = 0.2
    colsample_bytree = 0.6
    colsample_bylevel = 1.0

  XGBoost tuné (RandomizedSearch)
  MAE  :       15,253 MAD
  RMSE :       25,426 MAD
  R²   :       0.9051


In [32]:
# ============================================================
# STACKING — XGBoost tuné + RF → Ridge meta-modèle
# ============================================================
print("\n⏳ Stacking : XGBoost tuné + RandomForest → Ridge...")

rf_stack = RandomForestRegressor(
    n_estimators=300, max_features='sqrt',
    random_state=42, n_jobs=-1
)

xgb_stack = xgb.XGBRegressor(
    **search.best_params_,
    random_state=42, n_jobs=-1, eval_metric='mae'
)

stacking = StackingRegressor(
    estimators=[
        ('xgb', xgb_stack),
        ('rf',  rf_stack)
    ],
    final_estimator=Ridge(alpha=1.0),
    cv=5,
    n_jobs=-1
)

stacking.fit(X_train, y_train)
y_pred = stacking.predict(X_test)
resultats.append(evaluer("Stacking (XGBoost + RF → Ridge)", y_test, y_pred))


⏳ Stacking : XGBoost tuné + RandomForest → Ridge...

  Stacking (XGBoost + RF → Ridge)
  MAE  :       15,203 MAD
  RMSE :       25,416 MAD
  R²   :       0.9052


In [33]:
# ============================================================
# TABLEAU FINAL
# ============================================================
print("\n" + "="*60)
print("  TABLEAU FINAL — ÉTAPE 3")
print("="*60)
print(f"  {'Modèle':<42} {'MAE':>10} {'RMSE':>12} {'R²':>8}")
print(f"  {'-'*42} {'-'*10} {'-'*12} {'-'*8}")
for r in sorted(resultats, key=lambda x: x['r2'], reverse=True):
    print(f"  {r['nom']:<42} {r['mae']:>10,.0f} {r['rmse']:>12,.0f} {r['r2']:>8.4f}")

meilleur = max(resultats, key=lambda x: x['r2'])
print(f"\n🏆 Meilleur modèle final : {meilleur['nom']}")
print(f"   R² = {meilleur['r2']:.4f} | MAE = {meilleur['mae']:,.0f} MAD")


  TABLEAU FINAL — ÉTAPE 3
  Modèle                                            MAE         RMSE       R²
  ------------------------------------------ ---------- ------------ --------
  Stacking (XGBoost + RF → Ridge)                15,203       25,416   0.9052
  XGBoost tuné (RandomizedSearch)                15,253       25,426   0.9051
  XGBoost référence (étape 2)                    15,466       25,630   0.9036

🏆 Meilleur modèle final : Stacking (XGBoost + RF → Ridge)
   R² = 0.9052 | MAE = 15,203 MAD


In [34]:
# ============================================================
# IMPORTANCE DES FEATURES — Top 20
# ============================================================
print("\n📊 Top 20 features les plus importantes (XGBoost tuné) :")
feat_imp = pd.Series(
    best_xgb.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False).head(20)

for feat, imp in feat_imp.items():
    bar = '█' * int(imp * 500)
    print(f"  {feat:<40} {imp:.4f} {bar}")


📊 Top 20 features les plus importantes (XGBoost tuné) :
  Boite de vitesses                        0.0981 █████████████████████████████████████████████████
  Modèle_Jetta                             0.0230 ███████████
  Type de carburant_Essence                0.0190 █████████
  Marque_Mercedes-Benz                     0.0190 █████████
  Modèle_Tiguan                            0.0178 ████████
  Marque_Fiat                              0.0160 ███████
  Année-Modèle                             0.0149 ███████
  Modèle_Clio                              0.0149 ███████
  Modèle_Tucson                            0.0148 ███████
  Modèle_Accent                            0.0148 ███████
  Type de carburant_Diesel                 0.0145 ███████
  Marque_Volvo                             0.0142 ███████
  Marque_Audi                              0.0135 ██████
  Modèle_Giulietta                         0.0131 ██████
  Marque_BMW                               0.0125 ██████
  Puissance fiscale      

In [35]:
# ============================================================
# IMPORTANCE DES FEATURES — Top 20
# ============================================================
print("\n📊 Top 20 features les plus importantes (XGBoost tuné) :")
feat_imp = pd.Series(
    best_xgb.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False).head(20)

for feat, imp in feat_imp.items():
    bar = '█' * int(imp * 500)
    print(f"  {feat:<40} {imp:.4f} {bar}")


📊 Top 20 features les plus importantes (XGBoost tuné) :
  Boite de vitesses                        0.0981 █████████████████████████████████████████████████
  Modèle_Jetta                             0.0230 ███████████
  Type de carburant_Essence                0.0190 █████████
  Marque_Mercedes-Benz                     0.0190 █████████
  Modèle_Tiguan                            0.0178 ████████
  Marque_Fiat                              0.0160 ███████
  Année-Modèle                             0.0149 ███████
  Modèle_Clio                              0.0149 ███████
  Modèle_Tucson                            0.0148 ███████
  Modèle_Accent                            0.0148 ███████
  Type de carburant_Diesel                 0.0145 ███████
  Marque_Volvo                             0.0142 ███████
  Marque_Audi                              0.0135 ██████
  Modèle_Giulietta                         0.0131 ██████
  Marque_BMW                               0.0125 ██████
  Puissance fiscale      

In [36]:
# ============================================================
# SAUVEGARDE DU MEILLEUR MODÈLE FINAL
# ============================================================
# Choisir entre stacking et xgb tuné selon les résultats
modele_final = stacking if resultats[2]['r2'] > resultats[1]['r2'] else best_xgb
nom_final    = resultats[2]['nom'] if resultats[2]['r2'] > resultats[1]['r2'] else resultats[1]['nom']

with open('model_final.pkl', 'wb') as f:
    pickle.dump(modele_final, f)

with open('resultats_etape3.pkl', 'wb') as f:
    pickle.dump({
        'resultats': resultats,
        'best_params': search.best_params_,
        'feat_importance': feat_imp,
        'meilleur_nom': nom_final
    }, f)

print(f"\n✅ Étape 3 terminée !")
print(f"   Modèle final sauvegardé : model_final.pkl ({nom_final})")
print(f"   → Lance étape4_integration.py pour intégrer dans l'API Flask")


✅ Étape 3 terminée !
   Modèle final sauvegardé : model_final.pkl (Stacking (XGBoost + RF → Ridge))
   → Lance étape4_integration.py pour intégrer dans l'API Flask


In [37]:
# Script de vérification de compatibilité avant intégration
# Lance ce script AVANT de modifier api.py pour vérifier que tout est OK

import pickle
import numpy as np
import pandas as pd

print("="*60)
print("  VÉRIFICATION DE COMPATIBILITÉ — ÉTAPE 4")
print("="*60)

  VÉRIFICATION DE COMPATIBILITÉ — ÉTAPE 4


In [38]:
# 1. Chargement des nouveaux fichiers
print("\n⏳ Chargement des nouveaux fichiers...")
model    = pickle.load(open('model_final.pkl', 'rb'))
scaler   = pickle.load(open('scaler_v2.pkl', 'rb'))
features = pickle.load(open('features_v2.pkl', 'rb'))

print(f"  ✅ model_final.pkl    : {type(model).__name__}")
print(f"  ✅ scaler_v2.pkl      : {type(scaler).__name__}")
print(f"  ✅ features_v2.pkl    : {len(features)} colonnes")


⏳ Chargement des nouveaux fichiers...
  ✅ model_final.pkl    : StackingRegressor
  ✅ scaler_v2.pkl      : StandardScaler
  ✅ features_v2.pkl    : 663 colonnes


In [39]:
# 2. Colonnes numériques scalées
numerical_features = ['Année-Modèle', 'Kilométrage', 'Puissance fiscale', 'Age', 'Km_par_an']
print(f"\n  Colonnes scalées : {numerical_features}")
print(f"  Mean  : {scaler.mean_}")
print(f"  Scale : {scaler.scale_}")


  Colonnes scalées : ['Année-Modèle', 'Kilométrage', 'Puissance fiscale', 'Age', 'Km_par_an']
  Mean  : [2.01478886e+03 1.12481907e+11 7.04783490e+00 1.12111416e+01
 9.40121724e+09]
  Scale : [6.53025223e+00 9.54483587e+10 1.34963770e+00 6.53025223e+00
 8.39383491e+09]


In [40]:
# 3. Test de prédiction avec une entrée fictive
print("\n⏳ Test de prédiction avec une voiture fictive...")
print("   (Dacia Logan 2019, Essence, 80 000 km, Manuelle, Bon état)")


⏳ Test de prédiction avec une voiture fictive...
   (Dacia Logan 2019, Essence, 80 000 km, Manuelle, Bon état)


In [41]:
# Créer un vecteur de zéros
test_input = pd.DataFrame([np.zeros(len(features))], columns=features)

In [42]:
# Remplir les valeurs connues
test_input['Année-Modèle']     = 2019
test_input['Kilométrage']      = 80000
test_input['Puissance fiscale'] = 6
test_input['Nombre de portes'] = 5
test_input['État']             = 3   # Bon
test_input['Boite de vitesses'] = 0  # Manuelle
test_input['Age']       = 2026 - 2019          # = 7
test_input['Km_par_an'] = 80000 / (7 + 1)      # = 10 000
# One-hot
for col in features:
    if 'Marque_Dacia' in col:       test_input[col] = 1
    if 'Modèle_Logan' in col:       test_input[col] = 1
    if 'carburant_Essence' in col:  test_input[col] = 1
    if 'Origine_Dédouanée' in col:  test_input[col] = 1

In [43]:
# Appliquer le scaler
test_input[numerical_features] = scaler.transform(test_input[numerical_features])

In [44]:
# Prédire
prix_predit = model.predict(test_input)[0]
print(f"\n  ✅ Prix prédit : {prix_predit:,.0f} MAD")
print(f"  (Référence attendue pour Logan 2019 ~80k km : 70 000 – 100 000 MAD)")

print("\n✅ Vérification terminée — tu peux intégrer dans api.py")


  ✅ Prix prédit : 88,466 MAD
  (Référence attendue pour Logan 2019 ~80k km : 70 000 – 100 000 MAD)

✅ Vérification terminée — tu peux intégrer dans api.py


In [45]:
# models.py
from flask_sqlalchemy import SQLAlchemy
from datetime import datetime

db = SQLAlchemy()

class User(db.Model):
    id = db.Column(db.Integer, primary_key=True)
    username = db.Column(db.String(80), unique=True, nullable=False)
    email = db.Column(db.String(120), unique=True, nullable=False)
    password_hash = db.Column(db.String(256), nullable=False)
    is_admin = db.Column(db.Boolean, default=False)
    created_at = db.Column(db.DateTime, default=datetime.utcnow)

    predictions = db.relationship('PredictionHistory', backref='user', lazy=True)

class PredictionHistory(db.Model):
    id = db.Column(db.Integer, primary_key=True)
    user_id = db.Column(db.Integer, db.ForeignKey('user.id'), nullable=False)
    input_data = db.Column(db.Text)          # JSON des caractéristiques envoyées
    predicted_price = db.Column(db.Float)
    created_at = db.Column(db.DateTime, default=datetime.utcnow)
    description = db.Column(db.String(200))  # ex: "Renault Clio 2018"

In [46]:
# Lance ce script pour voir les noms EXACTS des équipements dans features_v2.pkl
import pickle

features = pickle.load(open('features_v2.pkl', 'rb'))

# Tous les noms possibles d'équipements dans le dataset
candidats = [
    'Climatisation', 'GPS', 'Caméra de recul', 'Jantes en alliage',
    'Jantes aluminium', 'Toit ouvrant', 'Sièges cuir', 'Radar de recul',
    'Vitres électriques', 'Verrouillage centralisé', 'Verrouillage centralisé à distance',
    'Régulateur de vitesse', 'ABS', 'Airbags', 'Direction assistée',
    'CD / MP3 / Bluetooth', 'CD/MP3/Bluetooth', 'ESP',
    'Limiteur de vitesse', 'Ordinateur de bord', 'Système de navigation/GPS',
    'Première main', 'Caméra 360°', 'Sieges cuir', 'Jantes',
]

print("="*55)
print("ÉQUIPEMENTS PRÉSENTS dans features_v2.pkl :")
print("="*55)
trouves = []
for c in candidats:
    if c in features:
        print(f"  ✅  '{c}'")
        trouves.append(c)
    else:
        print(f"  ❌  '{c}'  ← ABSENT")

print(f"\nTotal trouvés : {len(trouves)}")

# Chercher TOUS les features qui ressemblent à des équipements
print("\n" + "="*55)
print("SCAN COMPLET — features non-one-hot (probables équipements) :")
print("="*55)
prefixes = ['Marque_','Modèle_','Type de carburant_','Origine_',
            'Année','Kilom','Puissance','Nombre','État','Boite','Age','Km_']
for f in sorted(features):
    if not any(f.startswith(p) for p in prefixes):
        print(f"  → '{f}'")


ÉQUIPEMENTS PRÉSENTS dans features_v2.pkl :
  ✅  'Climatisation'
  ❌  'GPS'  ← ABSENT
  ✅  'Caméra de recul'
  ❌  'Jantes en alliage'  ← ABSENT
  ✅  'Jantes aluminium'
  ✅  'Toit ouvrant'
  ✅  'Sièges cuir'
  ✅  'Radar de recul'
  ✅  'Vitres électriques'
  ❌  'Verrouillage centralisé'  ← ABSENT
  ✅  'Verrouillage centralisé à distance'
  ✅  'Régulateur de vitesse'
  ✅  'ABS'
  ✅  'Airbags'
  ❌  'Direction assistée'  ← ABSENT
  ❌  'CD / MP3 / Bluetooth'  ← ABSENT
  ✅  'CD/MP3/Bluetooth'
  ✅  'ESP'
  ✅  'Limiteur de vitesse'
  ✅  'Ordinateur de bord'
  ✅  'Système de navigation/GPS'
  ❌  'Première main'  ← ABSENT
  ❌  'Caméra 360°'  ← ABSENT
  ❌  'Sieges cuir'  ← ABSENT
  ❌  'Jantes'  ← ABSENT

Total trouvés : 16

SCAN COMPLET — features non-one-hot (probables équipements) :
  → 'ABS'
  → 'Airbags'
  → 'CD/MP3/Bluetooth'
  → 'Caméra de recul'
  → 'Climatisation'
  → 'ESP'
  → 'Jantes aluminium'
  → 'Limiteur de vitesse'
  → 'Ordinateur de bord'
  → 'Radar de recul'
  → 'Régulateur de vit